
# VMD3 FMCW → Range–Azimuth → SAR Learning Notebook

This notebook is organized as a sequence of small, testable learning milestones.  
Do **not** move to the next milestone until the current output looks physically reasonable.

## Experimental runs

| Run | Measurement | Primary use |
|---|---|---|
| **A — Fixed-position phase test** | Radar at $x=0$; approximately 20 consecutive frames; sphere near boresight at about $R=0.84\ \text{m}$ | Decode the data, form a 1D range profile, form a beam-steered range–azimuth map, and verify frame-to-frame complex phase stability |
| **B — Background scan** | Sphere removed; 11 positions from $-5$ to $+5\ \text{cm}$ in $1\ \text{cm}$ increments; 3–5 frames per position | Characterize static clutter and create a background range-versus-position B-scan |
| **C — Centered sphere scan** | Sphere centered near $x_t=0$, $z_t\approx0.84\ \text{m}$; same 11 positions; about 5 frames per position | Primary SAR proof-of-concept dataset |
| **D — Offset sphere scan** | Sphere shifted to approximately $x_t=+3\ \text{cm}$; repeat Run C | Validate that the reconstructed target shifts by about $+3\ \text{cm}$ |

The sphere diameter is provisionally entered as **5 cm**. Confirm this from the lab notes before using it in any scattering or resolution analysis.

---

## Processing roadmap

1. Inspect the file format and identify the data axes.
2. Build the complex raw signal $I+jQ$.
3. Use one Run A frame, one beam, and one RX channel to plot raw RADC $I/Q$.
4. Perform a range FFT and make a 1D range profile.
5. Stack beam-steered range profiles to make a fixed-position range–azimuth heatmap.
6. Use Run A to verify complex phase stability across frames.
7. Use Runs B and C to make range-versus-mechanical-position B-scans.
8. Compare or subtract the background carefully.
9. Inspect the sphere phase history across aperture position.
10. Backproject a simulated point target.
11. Backproject Run C.
12. Use Run D to validate the reconstructed cross-range shift.

## Important distinction

A **range–azimuth map** from Run A uses the VMD3's commanded beam-steering angles.

A **range-versus-position B-scan** from Runs B–D uses the eleven mechanical radar positions.

A **focused SAR image** uses the complex measurements from those mechanical positions to estimate target coordinates.


In [ ]:

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

C0 = 299_792_458.0

# ---------------------------
# Experiment-level parameters
# ---------------------------
FC_HZ = 60e9
WAVELENGTH_M = C0 / FC_HZ

SPHERE_RANGE_M = 0.84
SPHERE_DIAMETER_M = 0.05       # Confirm from lab notes
APERTURE_POSITIONS_M = np.arange(-0.05, 0.0501, 0.01)  # 11 positions

# -----------------------------------------
# Radar parameters: fill these in from VMD3
# -----------------------------------------
ADC_SAMPLE_RATE_HZ = None
CHIRP_SLOPE_HZ_PER_S = None
CHIRP_BANDWIDTH_HZ = None
NFFT_RANGE = None
BEAM_ANGLES_DEG = None

# Update these folders after sharing the actual data layout.
DATA_ROOT = Path("data")
RUN_A_DIR = DATA_ROOT / "run_a"
RUN_B_DIR = DATA_ROOT / "run_b"
RUN_C_DIR = DATA_ROOT / "run_c"
RUN_D_DIR = DATA_ROOT / "run_d"

print(f"Wavelength at 60 GHz: {WAVELENGTH_M*1e3:.3f} mm")
print("Mechanical positions (cm):", APERTURE_POSITIONS_M * 100)



# Milestone 0 — Determine the file and array structure

Before writing radar processing code, determine:

- whether $I$ and $Q$ are stored in separate columns, interleaved columns, or as complex values;
- the number of frames;
- the number of commanded beam angles;
- the number and ordering of RX channels;
- the number of chirps per frame;
- the number of ADC samples per chirp;
- whether the supplied RFFT is complex or magnitude-only;
- the beam-angle list;
- the ADC sample rate $f_s$;
- the chirp slope $S=B/T_c$.

## Canonical internal array format

After loading, this notebook will use the following internal shape:

$$
\texttt{radc.shape} =
[\text{frame},\text{beam},\text{rx},\text{chirp},\text{adc sample}]
$$

Every entry should be complex:

$$
\texttt{radc} = I+jQ.
$$

The VMD3 files may use a different ordering. The loader's job is to convert the vendor ordering into this canonical ordering.


In [ ]:

def load_vmd3_radc(path: Path) -> np.ndarray:
    """
    Load one VMD3 RADC file and return a complex ndarray with shape:

        [frame, beam, rx, chirp, adc_sample]

    This is intentionally a placeholder until the actual files and metadata
    are inspected.

    Replace this function after determining:
      1. file type and delimiter,
      2. I/Q encoding,
      3. axis order,
      4. dimension sizes.
    """
    raise NotImplementedError(
        "Share one representative RADC file and its configuration/metadata "
        "so this loader can be completed."
    )


def describe_array(name: str, arr: np.ndarray) -> None:
    print(f"{name}.shape = {arr.shape}")
    print(f"{name}.dtype = {arr.dtype}")
    print(f"Complex: {np.iscomplexobj(arr)}")
    print(f"Finite fraction: {np.mean(np.isfinite(arr)):.6f}")



# Milestone 1 — Plot raw Run A $I/Q$

Select:

- one Run A frame;
- the beam nearest $0^\circ$;
- one RX channel;
- one chirp.

The selected RADC vector is

$$
x[n]=I[n]+jQ[n].
$$

Plot $I[n]$ and $Q[n]$ against fast-time sample number. At this stage, do not expect the waveform to look like a clean single sinusoid because the scene may contain several reflectors and leakage components.


In [ ]:

def plot_raw_iq(iq: np.ndarray, fs_hz: float | None = None) -> None:
    iq = np.asarray(iq)
    if iq.ndim != 1:
        raise ValueError("iq must be a one-dimensional complex ADC vector.")

    if fs_hz is None:
        x = np.arange(iq.size)
        xlabel = "ADC sample number"
    else:
        x = np.arange(iq.size) / fs_hz * 1e6
        xlabel = "Fast time within chirp (µs)"

    plt.figure(figsize=(10, 4.5))
    plt.plot(x, iq.real, label="I[n]")
    plt.plot(x, iq.imag, label="Q[n]")
    plt.xlabel(xlabel)
    plt.ylabel("ADC amplitude")
    plt.title("Raw complex FMCW beat signal")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


# After the loader is complete, the selection will look like:
#
# radc_a = load_vmd3_radc(RUN_A_DIR / "example_file")
# center_beam_index = np.argmin(np.abs(BEAM_ANGLES_DEG))
# iq_example = radc_a[0, center_beam_index, 0, 0, :]
# plot_raw_iq(iq_example, ADC_SAMPLE_RATE_HZ)



# Milestone 2 — Compute one complex range profile

For one chirp, first form the complex RADC vector

$$
x[n]=I[n]+jQ[n].
$$

Apply the same real window to both $I$ and $Q$, equivalently multiplying the complex vector:

$$
x_w[n]=w[n]x[n].
$$

Then calculate the range FFT:

$$
X[k]=\operatorname{FFT}\{x_w[n]\}.
$$

Each FFT bin $k$ corresponds to beat frequency

$$
f_k=\frac{k f_s}{N_{\mathrm{FFT}}},
$$

and therefore to range

$$
R_k=\frac{c f_k}{2S}.
$$

The array $X[k]$ is the **complex range profile**.  
Plot $|X[k]|$ in dB, but retain the original complex $X[k]$ for phase analysis and SAR.

## Beat-frequency sign warning

Different dechirp conventions can place physical targets at positive or negative beat frequency. If no target appears on the positive-frequency side, inspect the negative-frequency side before changing the processing.


In [ ]:

def db20(x: np.ndarray, floor_db: float = -120.0) -> np.ndarray:
    x = np.abs(np.asarray(x))
    peak = np.max(x)
    if peak <= 0:
        return np.full_like(x, floor_db, dtype=float)
    y = 20.0 * np.log10(x / peak + 1e-15)
    return np.maximum(y, floor_db)


def make_window(length: int, name: str = "hann") -> np.ndarray:
    name = name.lower()
    if name in {"hann", "hanning"}:
        return np.hanning(length)
    if name in {"rect", "rectangular", "none"}:
        return np.ones(length)
    raise ValueError(f"Unsupported window: {name}")


def range_fft_complex(
    iq: np.ndarray,
    fs_hz: float,
    slope_hz_per_s: float,
    nfft: int | None = None,
    window_name: str = "hann",
    remove_mean: bool = True,
    axis: int = -1,
    keep_positive: bool = True,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Compute a complex range FFT along `axis`.

    Returns
    -------
    spectrum : complex ndarray
        Complex range FFT.
    range_axis_m : ndarray
        Range coordinate for the retained FFT bins.
    beat_frequency_hz : ndarray
        Beat-frequency coordinate for the retained FFT bins.
    """
    iq = np.asarray(iq)
    if not np.iscomplexobj(iq):
        raise ValueError("iq must be complex, e.g. I + 1j*Q.")
    if fs_hz <= 0 or slope_hz_per_s == 0:
        raise ValueError("fs_hz must be positive and slope_hz_per_s nonzero.")

    ns = iq.shape[axis]
    if nfft is None:
        nfft = ns
    if nfft < ns:
        raise ValueError("nfft must be at least the number of ADC samples.")

    work = iq.astype(np.complex128, copy=True)

    if remove_mean:
        work = work - np.mean(work, axis=axis, keepdims=True)

    window = make_window(ns, window_name)
    reshape = [1] * work.ndim
    reshape[axis] = ns
    work *= window.reshape(reshape)

    spectrum = np.fft.fft(work, n=nfft, axis=axis)
    frequency_hz = np.fft.fftfreq(nfft, d=1.0 / fs_hz)

    if keep_positive:
        mask = frequency_hz >= 0
    else:
        mask = frequency_hz <= 0

    spectrum = np.take(spectrum, np.flatnonzero(mask), axis=axis)
    frequency_hz = frequency_hz[mask]

    # abs(slope) keeps the displayed range positive. Frequency sign is
    # retained separately in beat_frequency_hz.
    range_axis_m = C0 * np.abs(frequency_hz) / (2.0 * abs(slope_hz_per_s))
    return spectrum, range_axis_m, frequency_hz


def plot_range_profile(
    spectrum: np.ndarray,
    range_axis_m: np.ndarray,
    max_range_m: float | None = None,
    title: str = "1D range profile",
) -> None:
    spectrum = np.asarray(spectrum)
    if spectrum.ndim != 1:
        raise ValueError("spectrum must be one-dimensional.")

    mask = np.ones_like(range_axis_m, dtype=bool)
    if max_range_m is not None:
        mask &= range_axis_m <= max_range_m

    plt.figure(figsize=(10, 4.5))
    plt.plot(range_axis_m[mask], db20(spectrum[mask]))
    plt.xlabel("Range (m)")
    plt.ylabel("Normalized magnitude (dB)")
    plt.title(title)
    plt.grid(True)
    plt.tight_layout()
    plt.show()


# Example after loading:
#
# X, ranges_m, beat_hz = range_fft_complex(
#     iq_example,
#     fs_hz=ADC_SAMPLE_RATE_HZ,
#     slope_hz_per_s=CHIRP_SLOPE_HZ_PER_S,
#     nfft=NFFT_RANGE,
# )
# plot_range_profile(X, ranges_m, max_range_m=2.0)



# Milestone 3 — Run A fixed-position range–azimuth heatmap

Run A can produce two useful products without moving the radar:

## 1D product

Choose one beam, usually the beam nearest boresight, and plot magnitude versus range.

## 2D product

Compute one range profile for every commanded VMD3 beam angle and stack them:

$$
H(\theta,R)=|X(\theta,R)|.
$$

Display:

- horizontal axis: commanded beam angle;
- vertical axis: range;
- brightness: range-profile magnitude.

This is a **beam-steered range–azimuth heatmap**. It is not yet SAR.

### RX handling for this first map

Begin with one RX channel so the processing is transparent.

For a magnitude-only diagnostic map, a safe multi-RX option is noncoherent power summation:

$$
P(\theta,R)=\sum_m |X_m(\theta,R)|^2.
$$

Do not simply average complex RX channels unless the relative RX phases and steering calibration are understood.


In [ ]:

def combine_rx_power(spectrum_by_rx: np.ndarray, rx_axis: int) -> np.ndarray:
    """Noncoherent sum of RX-channel power."""
    return np.sum(np.abs(spectrum_by_rx) ** 2, axis=rx_axis)


def plot_range_azimuth(
    power_or_magnitude: np.ndarray,
    beam_angles_deg: np.ndarray,
    range_axis_m: np.ndarray,
    max_range_m: float | None = None,
    title: str = "Beam-steered range–azimuth heatmap",
) -> None:
    data = np.asarray(power_or_magnitude)
    if data.shape != (len(beam_angles_deg), len(range_axis_m)):
        raise ValueError(
            "Expected data shape [beam, range] matching the supplied axes."
        )

    range_mask = np.ones(len(range_axis_m), dtype=bool)
    if max_range_m is not None:
        range_mask &= range_axis_m <= max_range_m

    display = data[:, range_mask].T
    display_db = 10.0 * np.log10(
        display / np.max(display) + 1e-15
    )

    plt.figure(figsize=(10, 5.5))
    plt.imshow(
        display_db,
        origin="lower",
        aspect="auto",
        extent=[
            beam_angles_deg[0],
            beam_angles_deg[-1],
            range_axis_m[range_mask][0],
            range_axis_m[range_mask][-1],
        ],
        vmin=-45,
        vmax=0,
    )
    plt.xlabel("Commanded beam angle (deg)")
    plt.ylabel("Range (m)")
    plt.title(title)
    plt.colorbar(label="Normalized power (dB)")
    plt.tight_layout()
    plt.show()


# Conceptual Run A workflow:
#
# 1. Select one frame and one chirp:
#    iq_beam_rx_sample = radc_a[frame_index, :, :, chirp_index, :]
#    shape -> [beam, rx, adc_sample]
#
# 2. Range FFT along the ADC-sample axis:
#    X_a, ranges_m, _ = range_fft_complex(
#        iq_beam_rx_sample,
#        ADC_SAMPLE_RATE_HZ,
#        CHIRP_SLOPE_HZ_PER_S,
#        nfft=NFFT_RANGE,
#        axis=-1,
#    )
#    shape -> [beam, rx, range]
#
# 3. Sum RX power:
#    range_az_power = combine_rx_power(X_a, rx_axis=1)
#    shape -> [beam, range]
#
# 4. Plot:
#    plot_range_azimuth(
#        range_az_power,
#        BEAM_ANGLES_DEG,
#        ranges_m,
#        max_range_m=2.0,
#    )



# Milestone 4 — Run A frame-to-frame phase stability

This is the gate that determines whether straightforward coherent SAR is likely to work.

1. Use the beam nearest boresight.
2. Use one RX channel initially.
3. Form a complex range profile for every Run A frame.
4. Locate the sphere peak near $0.84\ \text{m}$.
5. Extract the same complex range bin from every frame.
6. Plot magnitude, wrapped phase, and unwrapped phase versus frame number.

For a stationary radar and target, the phase should be reasonably stable. Small drift is acceptable; large random frame-to-frame phase jumps are a warning.

Do not average frames coherently until this test is passed.


In [ ]:

def nearest_index(axis: np.ndarray, value: float) -> int:
    return int(np.argmin(np.abs(np.asarray(axis) - value)))


def plot_phase_stability(z: np.ndarray, title: str = "Run A phase stability") -> None:
    z = np.asarray(z)
    if z.ndim != 1 or not np.iscomplexobj(z):
        raise ValueError("z must be a one-dimensional complex sequence.")

    frame = np.arange(z.size)

    plt.figure(figsize=(10, 4.0))
    plt.plot(frame, np.abs(z), marker="o")
    plt.xlabel("Frame number")
    plt.ylabel("Magnitude")
    plt.title(f"{title}: magnitude")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 4.0))
    plt.plot(frame, np.angle(z), marker="o")
    plt.xlabel("Frame number")
    plt.ylabel("Wrapped phase (rad)")
    plt.title(f"{title}: wrapped phase")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 4.0))
    plt.plot(frame, np.unwrap(np.angle(z)), marker="o")
    plt.xlabel("Frame number")
    plt.ylabel("Unwrapped phase (rad)")
    plt.title(f"{title}: unwrapped phase")
    plt.grid(True)
    plt.tight_layout()
    plt.show()


# Conceptual extraction:
#
# X_frames shape: [frame, range]
# sphere_bin = nearest_index(ranges_m, SPHERE_RANGE_M)
# sphere_complex_by_frame = X_frames[:, sphere_bin]
# plot_phase_stability(sphere_complex_by_frame)



# Milestone 5 — Runs B and C range-versus-position B-scans

At every mechanical position $x_p$, form one complex range profile $X_p(R)$.

Stacking the **magnitudes** produces

$$
B(x_p,R)=|X_p(R)|.
$$

Display:

- horizontal axis: radar mechanical position;
- vertical axis: measured slant range;
- brightness: return magnitude.

This is an unfocused **B-scan**. The horizontal coordinate tells where the radar was, not where the target is.

## Frame averaging

- For a display-only B-scan, average power across frames:
  $$
  \overline{P}(R)=\frac{1}{M}\sum_m |X_m(R)|^2.
  $$
- For SAR, retain a complex profile at every position.
- Coherent frame averaging is allowed only after Run A demonstrates stable phase:
  $$
  \overline{X}(R)=\frac{1}{M}\sum_m X_m(R).
  $$

## Background subtraction

Complex Run C minus Run B subtraction is valid only if the two separate runs are mutually phase coherent and geometrically repeatable.

Start with side-by-side power maps. A conservative display-only difference is

$$
P_C-P_B.
$$

Do not use magnitude-only background subtraction as the final complex SAR input.


In [ ]:

def average_frame_power(X: np.ndarray, frame_axis: int = 0) -> np.ndarray:
    return np.mean(np.abs(X) ** 2, axis=frame_axis)


def average_frames_coherently(X: np.ndarray, frame_axis: int = 0) -> np.ndarray:
    return np.mean(X, axis=frame_axis)


def plot_bscan(
    power: np.ndarray,
    positions_m: np.ndarray,
    range_axis_m: np.ndarray,
    max_range_m: float | None = None,
    title: str = "Range-versus-position B-scan",
) -> None:
    power = np.asarray(power)
    if power.shape != (len(positions_m), len(range_axis_m)):
        raise ValueError("Expected power shape [position, range].")

    mask = np.ones(len(range_axis_m), dtype=bool)
    if max_range_m is not None:
        mask &= range_axis_m <= max_range_m

    image = power[:, mask].T
    image_db = 10.0 * np.log10(image / np.max(image) + 1e-15)

    plt.figure(figsize=(10, 5.5))
    plt.imshow(
        image_db,
        origin="lower",
        aspect="auto",
        extent=[
            positions_m[0] * 100,
            positions_m[-1] * 100,
            range_axis_m[mask][0],
            range_axis_m[mask][-1],
        ],
        vmin=-45,
        vmax=0,
    )
    plt.xlabel("Radar along-track position (cm)")
    plt.ylabel("Slant range (m)")
    plt.title(title)
    plt.colorbar(label="Normalized power (dB)")
    plt.tight_layout()
    plt.show()


# Expected products:
#
# run_b_power.shape -> [11 positions, range bins]
# run_c_power.shape -> [11 positions, range bins]
#
# plot_bscan(run_b_power, APERTURE_POSITIONS_M, ranges_m, max_range_m=2.0,
#            title="Run B background B-scan")
# plot_bscan(run_c_power, APERTURE_POSITIONS_M, ranges_m, max_range_m=2.0,
#            title="Run C centered-sphere B-scan")



# Milestone 6 — Understand the expected range migration and phase history

For a point target at $(x_t,z_t)$ and radar position $x_p$,

$$
R_p=\sqrt{(x_t-x_p)^2+z_t^2}.
$$

For the centered sphere with $z_t\approx0.84\ \text{m}$ and a $\pm5\ \text{cm}$ aperture,

$$
\Delta R_{\text{center-to-edge}}
=
\sqrt{0.84^2+0.05^2}-0.84
\approx1.49\ \text{mm}.
$$

This is likely smaller than one physical range-resolution cell, so the migration may be difficult to see in the magnitude B-scan.

At $60\ \text{GHz}$, however, the round-trip phase change is approximately

$$
\Delta\phi=\frac{4\pi\Delta R}{\lambda}\approx214^\circ.
$$

Therefore, the main SAR information in this experiment may appear as a smooth phase history even when the magnitude ridge looks nearly horizontal.


In [ ]:

def expected_point_target_history(
    positions_m: np.ndarray,
    target_x_m: float,
    target_z_m: float,
    wavelength_m: float,
    phase_sign: float = -1.0,
) -> tuple[np.ndarray, np.ndarray]:
    ranges = np.sqrt((positions_m - target_x_m) ** 2 + target_z_m ** 2)
    phase = phase_sign * 4.0 * np.pi * ranges / wavelength_m
    return ranges, phase


R_expected, phase_expected = expected_point_target_history(
    APERTURE_POSITIONS_M,
    target_x_m=0.0,
    target_z_m=SPHERE_RANGE_M,
    wavelength_m=WAVELENGTH_M,
)

print(
    "Expected center-to-edge range change:",
    (R_expected[0] - np.min(R_expected)) * 1e3,
    "mm",
)
print(
    "Expected center-to-edge phase change:",
    np.rad2deg(phase_expected[0] - phase_expected[len(phase_expected)//2]),
    "deg",
)

plt.figure(figsize=(9, 4))
plt.plot(APERTURE_POSITIONS_M * 100, R_expected * 100, marker="o")
plt.xlabel("Radar position (cm)")
plt.ylabel("Expected slant range (cm)")
plt.title("Centered point-target range history")
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 4))
plt.plot(
    APERTURE_POSITIONS_M * 100,
    np.unwrap(phase_expected),
    marker="o",
)
plt.xlabel("Radar position (cm)")
plt.ylabel("Expected unwrapped propagation phase (rad)")
plt.title("Centered point-target phase history")
plt.grid(True)
plt.tight_layout()
plt.show()



# Milestone 7 — Extract the measured Run C sphere phase history

Use the complex Run C range profiles:

$$
X[p,k].
$$

Initially, select the range bin nearest $0.84\ \text{m}$ and plot

$$
\angle X[p,k_s]
$$

against aperture position.

Later, improve the extraction by interpolating each profile at the geometry-predicted range $R_p$.

Compare the measured phase curvature with the expected point-target phase. The sign may be reversed because of the VMD3 dechirp convention.


In [ ]:

def interpolate_complex_1d(
    x_axis: np.ndarray,
    y_complex: np.ndarray,
    x_query: float,
) -> complex:
    real = np.interp(x_query, x_axis, np.real(y_complex), left=0.0, right=0.0)
    imag = np.interp(x_query, x_axis, np.imag(y_complex), left=0.0, right=0.0)
    return complex(real, imag)


def extract_geometry_tracked_samples(
    complex_profiles: np.ndarray,
    positions_m: np.ndarray,
    range_axis_m: np.ndarray,
    target_x_m: float,
    target_z_m: float,
) -> tuple[np.ndarray, np.ndarray]:
    """
    complex_profiles shape: [position, range]
    """
    if complex_profiles.shape != (len(positions_m), len(range_axis_m)):
        raise ValueError("Expected complex_profiles shape [position, range].")

    expected_ranges = np.sqrt(
        (positions_m - target_x_m) ** 2 + target_z_m ** 2
    )

    samples = np.array([
        interpolate_complex_1d(range_axis_m, complex_profiles[p], expected_ranges[p])
        for p in range(len(positions_m))
    ])
    return samples, expected_ranges


# Example:
#
# z_c, tracked_ranges = extract_geometry_tracked_samples(
#     run_c_complex_profiles,
#     APERTURE_POSITIONS_M,
#     ranges_m,
#     target_x_m=0.0,
#     target_z_m=SPHERE_RANGE_M,
# )
#
# plt.plot(APERTURE_POSITIONS_M*100, np.unwrap(np.angle(z_c)), marker="o")
# plt.xlabel("Radar position (cm)")
# plt.ylabel("Measured unwrapped phase (rad)")
# plt.grid(True)
# plt.show()



# Milestone 8 — Backproject a simulated point target first

For every image pixel $(x,z)$ and every radar position $x_p$, compute

$$
R_p(x,z)=\sqrt{(x-x_p)^2+z^2}.
$$

Interpolate the measured complex range profile at that expected range and compensate the expected propagation phase:

$$I(x,z)
=
\sum_p
\widetilde X_p\!\left(R_p(x,z)\right)
e^{s\,j4\pi R_p(x,z)/\lambda},
$$

where $s=+1$ or $-1$ depends on the signal convention.

The correct pixel produces coherent addition. Incorrect pixels do not.

Before using measured data, generate a synthetic point target with the same eleven positions and confirm that backprojection focuses it at the expected coordinate.


In [ ]:

def backproject(
    complex_profiles: np.ndarray,
    positions_m: np.ndarray,
    range_axis_m: np.ndarray,
    x_grid_m: np.ndarray,
    z_grid_m: np.ndarray,
    wavelength_m: float,
    phase_sign: float = +1.0,
    aperture_window: str = "hann",
) -> np.ndarray:
    """
    Basic near-field monostatic backprojection.

    Parameters
    ----------
    complex_profiles : [position, range] complex ndarray
    positions_m : radar x positions
    range_axis_m : range coordinate
    x_grid_m, z_grid_m : image coordinates
    wavelength_m : carrier wavelength
    phase_sign : try +1 and -1; the focusing convention depends on the data
    aperture_window : 'hann' or 'rectangular'
    """
    complex_profiles = np.asarray(complex_profiles)
    if complex_profiles.shape != (len(positions_m), len(range_axis_m)):
        raise ValueError("Expected complex_profiles shape [position, range].")
    if not np.iscomplexobj(complex_profiles):
        raise ValueError("Backprojection requires complex range profiles.")

    weights = make_window(len(positions_m), aperture_window)
    image = np.zeros((len(z_grid_m), len(x_grid_m)), dtype=np.complex128)

    for iz, z in enumerate(z_grid_m):
        for ix, x in enumerate(x_grid_m):
            total = 0.0j

            for p, radar_x in enumerate(positions_m):
                expected_range = np.sqrt((x - radar_x) ** 2 + z ** 2)

                measured = interpolate_complex_1d(
                    range_axis_m,
                    complex_profiles[p],
                    expected_range,
                )

                phase_correction = np.exp(
                    phase_sign * 1j * 4.0 * np.pi
                    * expected_range / wavelength_m
                )

                total += weights[p] * measured * phase_correction

            image[iz, ix] = total

    return image


def plot_sar_image(
    image: np.ndarray,
    x_grid_m: np.ndarray,
    z_grid_m: np.ndarray,
    title: str = "Backprojected SAR image",
    dynamic_range_db: float = 40.0,
) -> None:
    image_db = db20(image, floor_db=-dynamic_range_db)

    plt.figure(figsize=(9, 6))
    plt.imshow(
        image_db,
        origin="lower",
        aspect="auto",
        extent=[
            x_grid_m[0] * 100,
            x_grid_m[-1] * 100,
            z_grid_m[0],
            z_grid_m[-1],
        ],
        vmin=-dynamic_range_db,
        vmax=0,
    )
    plt.xlabel("Estimated cross-range x (cm)")
    plt.ylabel("Estimated down-range z (m)")
    plt.title(title)
    plt.colorbar(label="Normalized magnitude (dB)")
    plt.tight_layout()
    plt.show()



# Milestone 9 — Backproject Run C

Recommended first image grid:

- cross-range $x$: approximately $-10$ to $+10\ \text{cm}$;
- down-range $z$: a narrow region around the sphere, for example $0.70$ to $1.00\ \text{m}$.

Begin with:

- one beam near boresight;
- one RX channel;
- one complex profile per aperture position;
- no background subtraction;
- both phase-sign choices.

Then evaluate which sign focuses the response more tightly.

Only after that baseline works should you test:

- coherent frame averaging;
- background subtraction;
- multi-RX combination;
- aperture windows;
- amplitude/path-loss weighting;
- a wider image region.


In [ ]:

X_GRID_M = np.linspace(-0.10, 0.10, 241)
Z_GRID_M = np.linspace(0.70, 1.00, 301)

# Example once run_c_complex_profiles is available:
#
# image_plus = backproject(
#     run_c_complex_profiles,
#     APERTURE_POSITIONS_M,
#     ranges_m,
#     X_GRID_M,
#     Z_GRID_M,
#     WAVELENGTH_M,
#     phase_sign=+1.0,
# )
#
# image_minus = backproject(
#     run_c_complex_profiles,
#     APERTURE_POSITIONS_M,
#     ranges_m,
#     X_GRID_M,
#     Z_GRID_M,
#     WAVELENGTH_M,
#     phase_sign=-1.0,
# )
#
# plot_sar_image(image_plus, X_GRID_M, Z_GRID_M,
#                title="Run C backprojection: phase sign +1")
# plot_sar_image(image_minus, X_GRID_M, Z_GRID_M,
#                title="Run C backprojection: phase sign -1")



# Milestone 10 — Validate with Run D

Backproject Run D using the same:

- file-decoding method;
- chirp and beam selection;
- RX channel;
- range-FFT settings;
- frame-averaging method;
- image grid;
- phase convention;
- display dynamic range.

Find the strongest reconstructed response in Runs C and D.

The Run D peak should move approximately

$$
\Delta x \approx +3\ \text{cm}.
$$

This is stronger evidence of successful cross-range reconstruction than simply obtaining a bright image from Run C.


In [ ]:

def peak_coordinate(
    image: np.ndarray,
    x_grid_m: np.ndarray,
    z_grid_m: np.ndarray,
) -> tuple[float, float, float]:
    iz, ix = np.unravel_index(np.argmax(np.abs(image)), image.shape)
    return x_grid_m[ix], z_grid_m[iz], float(np.abs(image[iz, ix]))


# Example:
#
# x_c, z_c, amp_c = peak_coordinate(image_c, X_GRID_M, Z_GRID_M)
# x_d, z_d, amp_d = peak_coordinate(image_d, X_GRID_M, Z_GRID_M)
#
# print(f"Run C peak: x={x_c*100:.2f} cm, z={z_c:.3f} m")
# print(f"Run D peak: x={x_d*100:.2f} cm, z={z_d:.3f} m")
# print(f"Measured cross-range shift: {(x_d-x_c)*100:.2f} cm")



# Suggested notebook working order

## Session 1 — Decode one Run A file

Output checklist:

- [ ] file loads without errors;
- [ ] array axes are identified;
- [ ] $I+jQ$ is formed correctly;
- [ ] one raw RADC waveform is plotted;
- [ ] radar configuration parameters are entered.

## Session 2 — One-dimensional range processing

- [ ] one complex FFT is calculated;
- [ ] range bins are converted to meters;
- [ ] a peak appears near the expected sphere range;
- [ ] rectangular and Hann windows are compared;
- [ ] VMD3 RFFT and notebook FFT are compared if possible.

## Session 3 — Fixed-position two-dimensional beam scan

- [ ] one range profile is produced for every beam angle;
- [ ] the range–azimuth heatmap is plotted;
- [ ] the sphere response is near boresight and near $0.84\ \text{m}$;
- [ ] one-RX and noncoherent multi-RX maps are compared.

## Session 4 — Coherence gate

- [ ] complex sphere-bin phase is extracted across the 20 Run A frames;
- [ ] frame-to-frame phase stability is quantified;
- [ ] a justified frame-averaging method is selected.

## Session 5 — Mechanical aperture diagnostics

- [ ] Run B background B-scan;
- [ ] Run C sphere B-scan;
- [ ] Run C minus Run B display comparison;
- [ ] measured phase versus aperture position.

## Session 6 — Image formation

- [ ] simulated point-target backprojection;
- [ ] Run C backprojection;
- [ ] phase-sign selection;
- [ ] Run D backprojection;
- [ ] measured Run C to Run D cross-range shift.

---

# Data needed next

To complete the loader and first processing cells, provide:

1. one representative Run A RADC file;
2. one matching RFFT file;
3. one DONE or metadata/configuration file;
4. the beam-angle list or enough metadata to reconstruct it;
5. the VMD3 chirp settings, especially ADC sample rate, chirp duration, bandwidth, and slope;
6. a short explanation of how the files are named across frame, beam, RX, chirp, and mechanical position.

The immediate next goal is only:

$$
\boxed{\text{one Run A file} \rightarrow \text{one correct complex 1D range profile}}
$$

Everything else in this notebook builds from that result.
